# **Práctica Calificada 2 — Agentes Inteligentes**
## Construcción e Implementación de una Herramienta Interactiva de ML


| Campo | Detalle |
|---|---|
| **Universidad** | Universidad San Ignacio de Loyola (USIL) |
| **Curso** | Agentes Inteligentes — SFW52059 |
| **Semestre** | 2026-1 |
| **Notebook** | 1 de 2 — Análisis Exploratorio de Datos (EDA) |
| **Dataset** | Life Expectancy (WHO) — Kaggle |
| **Fuente** | https://www.kaggle.com/datasets/kumarajarshi/life-expectancy-who |
| **Tarea ML** | Regresión supervisada |
| **Herramienta** | Python · pandas · matplotlib · seaborn · scikit-learn |


## Contenido de este notebook

| Celda | Sección del informe | Descripción |
|---|---|---|
| 1 | — | Importación de librerías |
| 2 | — | Carga del dataset |
| 3 | 1.2 | Estructura y tipos de datos |
| 4 | 1.2 | Estadísticos descriptivos |
| 5 | 1.2 | Valores nulos y duplicados |
| 6 | 1.2 | Preprocesamiento base (imputación) |
| 7 | 1.3 | Gráfico 1 — Scatter: GDP vs Life expectancy |
| 8 | 1.2 / 1.3 | Gráfico 2 — Heatmap de correlación |
| 9 | 1.2 / 1.3 | Gráfico 3 — Boxplot por tipo de país |
| 10 | 1.3 | Gráfico 4 — Serie temporal 2000–2015 |
| 11 | 1.2 | Análisis de outliers (método IQR) |


**Instrucciones de ejecución:**
1. Sube el archivo `Life Expectancy Data.csv` usando el ícono de carpeta del panel izquierdo.
2. Ejecuta las celdas en orden secuencial.
3. O usa: **Entorno de ejecución → Reiniciar sesión y ejecutar todo**.

In [ ]:

# ─────────────────────────────────────────────
# Instalación e importaciones
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Estilo global de gráficos
plt.rcParams.update({
    'figure.dpi': 150,
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f8f8',
    'axes.grid': True,
    'grid.color': 'white',
    'grid.linewidth': 1.2,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

TEAL   = '#1D9E75'
CORAL  = '#D85A30'
PURPLE = '#7F77DD'
AMBER  = '#BA7517'
GRAY   = '#888780'

print(" Librerías importadas correctamente.")

In [ ]:
# ─────────────────────────────────────────────
# Carga del dataset
# ─────────────────────────────────────────────

df_raw = pd.read_csv('Life Expectancy Data.csv')

# Limpieza de nombres de columnas (espacios extra)
df_raw.columns = df_raw.columns.str.strip()

print(f" Dataset cargado: {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas")
print(f"\n Columnas disponibles:\n{list(df_raw.columns)}")
print(f"\n Primeras 3 filas:")
df_raw.head(3)

In [ ]:
# ─────────────────────────────────────────────
# Tipos de datos y estructura
# ─────────────────────────────────────────────
print("=" * 60)
print("ESTRUCTURA DEL DATASET")
print("=" * 60)
print(df_raw.dtypes.to_string())
print(f"\nMemoria usada: {df_raw.memory_usage(deep=True).sum() / 1024:.1f} KB")

In [ ]:
# ─────────────────────────────────────────────
# Estadísticos descriptivos completos
# ─────────────────────────────────────────────
print("=" * 60)
print("ESTADÍSTICOS DESCRIPTIVOS — VARIABLES NUMÉRICAS")
print("=" * 60)

numericas = df_raw.select_dtypes(include='number').columns.tolist()
desc = df_raw[numericas].describe().T
desc['median'] = df_raw[numericas].median()
desc = desc[['count', 'mean', '50%', 'std', 'min', 'max', 'median']]
desc.columns = ['n', 'media', 'p50', 'desv_std', 'mín', 'máx', 'mediana']
desc = desc.round(3)

print(desc.to_string())

In [ ]:
# ─────────────────────────────────────────────
# Valores nulos y duplicados
# ─────────────────────────────────────────────
print("=" * 60)
print("VALORES NULOS Y DUPLICADOS")
print("=" * 60)

nulos = pd.DataFrame({
    'Nulos absolutos': df_raw.isnull().sum(),
    'Porcentaje (%)': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
})
nulos = nulos[nulos['Nulos absolutos'] > 0].sort_values('Porcentaje (%)', ascending=False)

print(f"\nVariables con valores nulos ({len(nulos)} de {df_raw.shape[1]}):\n")
print(nulos.to_string())

duplicados = df_raw.duplicated().sum()
print(f"\nFilas duplicadas: {duplicados}")

**CRITERIO DE DECISIÓN APLICADO:**

*   Se imputará con la MEDIANA de cada variable (robusta ante outliers).
*   No se eliminarán filas para preservar representatividad.
*   Country y Year se excluirán del modelo, no requieren imputación.

In [ ]:
# ─────────────────────────────────────────────
# Preprocesamiento base para EDA
# ─────────────────────────────────────────────
df = df_raw.copy()

# Imputar nulos con mediana
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

# Confirmar sin nulos
assert df.isnull().sum().sum() == 0, "Aún hay nulos"
print(f" Imputación completada. Nulos restantes: {df.isnull().sum().sum()}")
print(f" Shape final para EDA: {df.shape}")

In [ ]:
# ─────────────────────────────────────────────
# VISUALIZACIÓN 1
# Scatter: GDP per cápita vs Life Expectancy
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

colores = {
    'Developed':   TEAL,
    'Developing':  CORAL
}

for status, grp in df.groupby('Status'):
    ax.scatter(
        grp['GDP'],
        grp['Life expectancy'],
        c=colores[status],
        alpha=0.45,
        s=22,
        label=status,
        edgecolors='none'
    )

# Línea de tendencia global (log-transform del GDP)
mask = df['GDP'] > 0
gdp_log = np.log(df.loc[mask, 'GDP'])
life    = df.loc[mask, 'Life expectancy']
slope, intercept, r, p, _ = stats.linregress(gdp_log, life)
x_range = np.linspace(gdp_log.min(), gdp_log.max(), 200)
ax.plot(np.exp(x_range), slope * x_range + intercept,
        color=PURPLE, linewidth=2, linestyle='--', label=f'Tendencia (r={r:.2f})')

ax.set_xscale('log')
ax.set_xlabel('PIB per cápita (USD, escala log)', fontsize=12)
ax.set_ylabel('Expectativa de vida (años)', fontsize=12)
ax.set_title('Gráfico 1 — Relación entre PIB per cápita y expectativa de vida\npor tipo de país (2000–2015)',
             fontsize=13, fontweight='bold', pad=14)
ax.legend(fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('grafico1_scatter_gdp_life.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Gráfico 1 guardado como 'grafico1_scatter_gdp_life.png'")

In [ ]:
# ─────────────────────────────────────────────
# VISUALIZACIÓN 2
# Heatmap de correlación
# ─────────────────────────────────────────────
cols_heatmap = [
    'Life expectancy', 'Adult Mortality', 'infant deaths',
    'Alcohol', 'percentage expenditure', 'Hepatitis B',
    'Measles', 'BMI', 'under-five deaths', 'Polio',
    'Total expenditure', 'Diphtheria', 'HIV/AIDS',
    'GDP', 'Population', 'thinness  1-19 years',
    'Income composition of resources', 'Schooling'
]

corr = df[cols_heatmap].corr()

# Renombrar para el gráfico (etiquetas más cortas)
labels = {
    'Life expectancy': 'Life expectancy',
    'Adult Mortality': 'Adult mortality',
    'infant deaths': 'Infant deaths',
    'Alcohol': 'Alcohol',
    'percentage expenditure': 'Health expenditure %',
    'Hepatitis B': 'Hepatitis B',
    'Measles': 'Measles',
    'BMI': 'BMI',
    'under-five deaths': 'Under-5 deaths',
    'Polio': 'Polio',
    'Total expenditure': 'Total expenditure',
    'Diphtheria': 'Diphtheria',
    'HIV/AIDS': 'HIV/AIDS',
    'GDP': 'GDP',
    'Population': 'Population',
    'thinness  1-19 years': 'Thinness 1–19y',
    'Income composition of resources': 'ICOR',
    'Schooling': 'Schooling'
}
corr.index   = [labels[c] for c in corr.index]
corr.columns = [labels[c] for c in corr.columns]

fig, ax = plt.subplots(figsize=(13, 11))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.3,
    linecolor='white',
    annot_kws={'size': 8},
    ax=ax,
    square=True,
    cbar_kws={'shrink': 0.7, 'label': 'Correlación de Pearson'}
)

ax.set_title('Gráfico 2 — Matriz de correlación de Pearson\nDataset Life Expectancy (WHO)',
             fontsize=13, fontweight='bold', pad=16)
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', rotation=0,  labelsize=9)

plt.tight_layout()
plt.savefig('grafico2_heatmap_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Gráfico 2 guardado como 'grafico2_heatmap_correlacion.png'")

# Imprimir correlaciones con el target
print("\n TOP correlaciones con 'Life expectancy':")
print(corr['Life expectancy'].drop('Life expectancy').sort_values().to_string())

In [ ]:
# ─────────────────────────────────────────────
# VISUALIZACIÓN 3
# Boxplot: Life expectancy por Status
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Boxplot principal
bp = axes[0].boxplot(
    [df[df['Status'] == s]['Life expectancy'].values for s in ['Developed', 'Developing']],
    labels=['Developed', 'Developing'],
    patch_artist=True,
    medianprops=dict(color='white', linewidth=2.5),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2),
    flierprops=dict(marker='o', markersize=3, alpha=0.4)
)
bp['boxes'][0].set_facecolor(TEAL)
bp['boxes'][0].set_alpha(0.75)
bp['boxes'][1].set_facecolor(CORAL)
bp['boxes'][1].set_alpha(0.75)
bp['fliers'][0].set_markerfacecolor(TEAL)
bp['fliers'][1].set_markerfacecolor(CORAL)

axes[0].set_ylabel('Expectativa de vida (años)', fontsize=11)
axes[0].set_title('Distribución por tipo de país', fontsize=12, fontweight='bold')

# Añadir medias como puntos
for i, status in enumerate(['Developed', 'Developing'], 1):
    media = df[df['Status'] == status]['Life expectancy'].mean()
    axes[0].plot(i, media, 'D', color='white',
                 markeredgecolor='#333', markersize=8, zorder=5,
                 label=f'Media {status}: {media:.1f}')
axes[0].legend(fontsize=9)

# Histograma superpuesto
for status, color in [('Developed', TEAL), ('Developing', CORAL)]:
    axes[1].hist(
        df[df['Status'] == status]['Life expectancy'],
        bins=25, alpha=0.6, color=color, label=status,
        edgecolor='white', linewidth=0.5
    )
axes[1].set_xlabel('Expectativa de vida (años)', fontsize=11)
axes[1].set_ylabel('Frecuencia', fontsize=11)
axes[1].set_title('Distribución de frecuencias', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

fig.suptitle('Gráfico 3 — Expectativa de vida por tipo de país (Developed vs Developing)',
             fontsize=13, fontweight='bold', y=1.01)

plt.tight_layout()
plt.savefig('grafico3_boxplot_status.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Gráfico 3 guardado como 'grafico3_boxplot_status.png'")

In [ ]:
# ─────────────────────────────────────────────
# VISUALIZACIÓN 4
# Serie temporal: promedio por año y Status
# ─────────────────────────────────────────────
serie = df.groupby(['Year', 'Status'])['Life expectancy'].mean().reset_index()

fig, ax = plt.subplots(figsize=(11, 5))

for status, color in [('Developed', TEAL), ('Developing', CORAL)]:
    sub = serie[serie['Status'] == status]
    ax.plot(sub['Year'], sub['Life expectancy'],
            color=color, linewidth=2.5, marker='o',
            markersize=6, label=status)
    ax.fill_between(sub['Year'], sub['Life expectancy'],
                    alpha=0.08, color=color)

    # Etiqueta al final de la línea
    last = sub.iloc[-1]
    ax.annotate(f"{last['Life expectancy']:.1f} años",
                xy=(last['Year'], last['Life expectancy']),
                xytext=(8, 0), textcoords='offset points',
                fontsize=10, color=color, fontweight='bold')

# Brecha inicial y final
inicio_dev  = serie[(serie['Year'] == 2000) & (serie['Status'] == 'Developing')]['Life expectancy'].values[0]
inicio_dved = serie[(serie['Year'] == 2000) & (serie['Status'] == 'Developed')]['Life expectancy'].values[0]
fin_dev  = serie[(serie['Year'] == 2015) & (serie['Status'] == 'Developing')]['Life expectancy'].values[0]
fin_dved = serie[(serie['Year'] == 2015) & (serie['Status'] == 'Developed')]['Life expectancy'].values[0]
ax.annotate(f"Brecha 2000: {inicio_dved - inicio_dev:.1f} años",
            xy=(2000, (inicio_dev + inicio_dved) / 2),
            xytext=(-60, 0), textcoords='offset points',
            fontsize=9, color=GRAY, style='italic')
ax.annotate(f"Brecha 2015: {fin_dved - fin_dev:.1f} años",
            xy=(2015, (fin_dev + fin_dved) / 2),
            xytext=(-130, 0), textcoords='offset points',
            fontsize=9, color=GRAY, style='italic')

ax.set_xlabel('Año', fontsize=12)
ax.set_ylabel('Expectativa de vida promedio (años)', fontsize=12)
ax.set_title('Gráfico 4 — Evolución temporal de la expectativa de vida promedio\npor tipo de país (2000–2015)',
             fontsize=13, fontweight='bold', pad=14)
ax.set_xticks(serie['Year'].unique())
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=11)
ax.set_xlim(1999, 2016.5)

plt.tight_layout()
plt.savefig('grafico4_serie_temporal.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Gráfico 4 guardado como 'grafico4_serie_temporal.png'")

In [ ]:
# ─────────────────────────────────────────────
# Análisis de outliers (IQR)
# ─────────────────────────────────────────────
print("=" * 60)
print("DETECCIÓN DE OUTLIERS — MÉTODO IQR")
print("=" * 60)

vars_outlier = [
    'Life expectancy', 'Adult Mortality', 'infant deaths',
    'GDP', 'Population', 'HIV/AIDS', 'Schooling',
    'Income composition of resources'
]

resumen_outliers = []
for col in vars_outlier:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    pct   = n_out / len(df) * 100
    resumen_outliers.append({
        'Variable': col,
        'Q1': round(Q1, 2),
        'Q3': round(Q3, 2),
        'IQR': round(IQR, 2),
        'Límite inf.': round(lower, 2),
        'Límite sup.': round(upper, 2),
        'N outliers': n_out,
        '% outliers': round(pct, 2)
    })

df_out = pd.DataFrame(resumen_outliers)
print(df_out.to_string(index=False))

**CRITERIO APLICADO:**
*   Los outliers se CONSERVAN en el dataset.
*   Corresponden a países reales en situaciones extremas (epidemias VIH/SIDA, conflictos armados, extrema pobreza).
*   Eliminarlos sesgaría el modelo hacia países "promedio" y reduciría la representatividad global del dataset.
*   Los modelos de árbol (Random Forest) son robustos ante outliers.